# Intelligent Customer Contact Optimization System
## Consolidated Pipeline Notebook (Notebooks 01–09)

This single notebook consolidates the full project — from raw data understanding through
segmentation, synthetic campaign history, and predictive modeling — into one reproducible
pipeline, with documentation and **bug fixes** applied along the way.

**Pipeline stages**

1. Data Understanding & Quality Assessment
2. Statistical Validation & Business Hypothesis Testing
3. Feature Engineering
4. Customer Segmentation & Persona Discovery
5. Synthetic Campaign History Generation
6. Best Time-to-Call Model
7. Look-Alike Model (Cold Start)
8. Channel Affinity & Response Propensity Models
9. Executive Business Report

> This notebook is intentionally left **unexecuted** (no run outputs), matching the state of
> the original notebooks supplied. `xgboost`, `lightgbm`, `statsmodels`, and `plotly` are not
> installed in this environment, so cells that depend on them are written correctly but not
> run here. Run top-to-bottom in an environment with those packages installed.

## ⚠️ Bug Fixes Applied in This Consolidation

Going through the original notebooks end-to-end surfaced two **material bugs** and a few smaller
issues. All are fixed in the code below; details are noted again at the relevant section.

| # | Location | Issue | Impact | Fix |
|---|----------|-------|--------|-----|
| 1 | **Notebook 01** | `duration` (call length) is correctly *identified* as a leakage variable in the markdown commentary, but the code never actually drops it before saving `bank_clean.csv`. | `duration` silently flows through into `bank_feature_engineered.csv` and `bank_segmented.csv`. It isn't used as a model feature downstream, but its presence contradicts the notebook's own stated methodology and is a landmine for future feature lists (`X = df[some_list_that_globs_all_columns]`). | Drop `duration` into a separate leakage-audit dataframe, save the **production** `bank_clean.csv` without it, and keep an `_experimental` copy with it retained, matching the notebook's own stated intent. |
| 2 | **Notebook 05** | `for _, row in customers.iterrows(): ... for _ in range(n): ... 'customer_id': _` — the inner loop variable `_` **shadows** the outer `_` (the customer's row index from `iterrows()`). Every interaction record ends up tagged with the *inner loop counter* (0..n-1) instead of the real customer id. | Confirmed on the supplied `synthetic_campaign_history.csv`: it contains only **7 distinct `customer_id` values** (0–6) across 225,985 rows, instead of ~45,211. Every downstream join in Notebooks 07/08/09 (`merge(hist, on='customer_id')`) is therefore matching on the wrong key, silently corrupting the look-alike, channel-affinity, and response models. | Rename the outer loop variable to `cust_idx` and the inner loop variable to `_i`, and use `cust_idx` as `customer_id`. |
| 3 | **Notebook 06** (original) | Only a placeholder stub was supplied (`# See chat for full code`), referencing an "XGBoost LabelEncoder wrapper" fix. | No working Best-Time-to-Call model existed. | Rebuilt Notebook 06 from scratch (Part 7 below) predicting `best_time_slot` from interaction/customer features, with a LabelEncoder wrapper around `XGBClassifier` (XGBoost's sklearn API expects `0..K-1` integer targets for multiclass; passing raw string labels like `"Monday Morning"` raises `label must be in [0, num_class)` on many xgboost versions). |
| 4 | **Notebooks 07 & 08** | Same string-label issue: `XGBClassifier` fit directly on string targets (`best_time_slot`, `channel`). | Works on very recent xgboost releases but throws on many pinned versions; fragile either way. | Wrapped every `XGBClassifier` multiclass fit with a `LabelEncoder`-based helper (`fit_xgb_multiclass` / `predict_xgb_multiclass`) so behavior is version-independent. RandomForest and LightGBM are left as-is since both natively accept string labels. |
| 5 | **Notebook 07** | `cold` (customers with no interaction history) will always be empty, because Notebook 05 now gives every customer 3–7 interactions. | The cold-start branch prints "No cold-start customers" and never actually exercises the look-alike logic. | Kept the logic (it's correct once real cold-start customers exist, e.g. new-to-bank customers not yet contacted) but added a synthetic hold-out split so the look-alike path is demonstrably exercised — see Part 8. |

Everything else (feature engineering formulas, clustering, persona mapping, report structure) was logically sound and is preserved as-is, just re-documented.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind, shapiro, zscore

from sklearn.preprocessing import LabelEncoder, RobustScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, classification_report, accuracy_score,
                              f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from scipy.cluster.hierarchy import linkage, dendrogram

# Optional heavy dependencies - imported defensively so the notebook can still be
# read/authored in environments where they are not installed.
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed - XGBoost cells will be skipped at runtime.")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("lightgbm not installed - LightGBM cells will be skipped at runtime.")

try:
    import statsmodels.api as sm
    from statsmodels.stats.multitest import multipletests
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    HAS_SM = True
except ImportError:
    HAS_SM = False
    print("statsmodels not installed - VIF/multiple-testing cells will be skipped at runtime.")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
plt.style.use("ggplot")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
def fit_xgb_multiclass(X_train, y_train, **xgb_kwargs):
    """Fit an XGBClassifier on possibly-non-numeric multiclass labels.

    XGBoost's sklearn API expects integer targets in [0, num_class) for
    multiclass objectives. This helper label-encodes y before fitting and
    returns both the fitted model and the encoder needed to invert
    predictions back to the original class names.
    """
    le = LabelEncoder()
    y_enc = le.fit_transform(y_train)
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(le.classes_),
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        **xgb_kwargs,
    )
    model.fit(X_train, y_enc)
    return model, le


def predict_xgb_multiclass(model, le, X):
    """Predict with a model trained via fit_xgb_multiclass and decode labels back."""
    return le.inverse_transform(model.predict(X))

## 1. Data Understanding & Quality Assessment
*(Notebook 01)*

Objectives: understand the structure of the dataset, assess data quality, detect
missing values/duplicates, explore demographics and campaign outcomes, and identify
potential data leakage before feature engineering begins.

In [ ]:
df = pd.read_csv("bank-full.csv", sep=";")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.info()
df.sample(10)

In [ ]:
data_dictionary = pd.DataFrame({
    "Feature": ["age","job","marital","education","default","balance","housing","loan",
                "contact","day","month","duration","campaign","pdays","previous","poutcome","y"],
    "Description": [
        "Customer age","Occupation","Marital status","Education","Credit default",
        "Average yearly balance","Housing loan","Personal loan","Contact type",
        "Last contact day","Last contact month","Last call duration (seconds)",
        "Number of contacts in this campaign","Days since previous contact",
        "Number of previous contacts","Previous campaign outcome","Campaign response"
    ]
})
data_dictionary

In [ ]:
missing = pd.DataFrame(df.isnull().sum(), columns=["Missing"])
missing["Percentage"] = missing["Missing"] / len(df) * 100
display(missing.sort_values("Percentage", ascending=False))

duplicates = df.duplicated().sum()
print(f"Duplicate Rows: {duplicates}")
df = df.drop_duplicates()

df.describe().T

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="y")
plt.title("Campaign Response Distribution")
plt.show()

response_rate = df["y"].value_counts(normalize=True) * 100
response_rate

**Business Insight:** the response variable is highly imbalanced — most customers do
not subscribe to the offered product, consistent with typical marketing conversion rates
below 20%. This imbalance is handled downstream via macro-F1 / ROC-AUC evaluation rather
than raw accuracy.

In [ ]:
numerical = df.select_dtypes(include=np.number).columns
categorical = df.select_dtypes(include="object").columns.drop("y")

for col in numerical:
    plt.figure(figsize=(7,4))
    sns.histplot(df[col], kde=True)
    plt.title(col)
    plt.show()

for col in numerical:
    plt.figure(figsize=(7,2))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()

In [ ]:
for col in categorical:
    plt.figure(figsize=(10,4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.xticks(rotation=45)
    plt.title(col)
    plt.show()

for col in categorical:
    plt.figure(figsize=(10,4))
    temp = pd.crosstab(df[col], df["y"], normalize="index")
    temp["yes"].sort_values().plot(kind="barh")
    plt.title(f"Response Rate by {col}")
    plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df[numerical].corr(), annot=True, cmap="coolwarm")
plt.show()

z_scores = np.abs(zscore(df[numerical]))
outliers = (z_scores > 3).sum(axis=0)
pd.DataFrame({"Feature": numerical, "Outliers": outliers})

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="y", y="duration")
plt.show()

**Important Observation — Data Leakage:** `duration` (last call length) is only known
*after* the call has ended. Since the business objective is deciding *whether/when to
contact* a customer *before* the call happens, `duration` is not available at prediction
time and must **not** be used as a model feature.

> **Fix applied:** the original notebook stated this but never actually dropped the
> column before saving `bank_clean.csv`, letting `duration` silently ride along into every
> downstream file. Below we now explicitly split into a production frame (no `duration`)
> and an experimental frame (with `duration`, for leakage-effect comparison only).

In [ ]:
business_insights = pd.DataFrame({
    "Finding": [
        "Response rate is highly imbalanced",
        "Cellular dominates telephone contact",
        "Previous successful campaigns increase conversion",
        "Duration is a leakage variable",
        "Balance shows high variance / long right tail",
        "Campaign frequency may indicate customer fatigue",
    ]
})
business_insights

In [ ]:
# --- FIX #1: actually remove the leakage variable before saving the "clean" dataset ---
df_experimental = df.copy()               # retains `duration`, for leakage-impact analysis only
df_clean = df.drop(columns=["duration"])  # production dataset used by every downstream notebook

df_clean.to_csv("bank_clean.csv", index=False)
df_experimental.to_csv("bank_clean_experimental_with_duration.csv", index=False)
print("Saved bank_clean.csv (production, duration removed) "
      "and bank_clean_experimental_with_duration.csv (duration retained).")

## 2. Statistical Validation & Business Hypothesis Testing
*(Notebook 02)*

Validate which features are statistically associated with the response variable (`y`)
before committing to them in feature engineering: chi-square tests for categoricals,
Welch's t-test / Mann-Whitney U for numerics (chosen per-feature via a Shapiro-Wilk
normality check), and a correlation/VIF pass for multicollinearity.

In [ ]:
df = pd.read_csv("bank_clean.csv")
df["target"] = df["y"].map({"yes": 1, "no": 0})

In [ ]:
cat_cols = ["job","marital","education","default","housing","loan","contact","month","poutcome"]
results = []
for c in cat_cols:
    tab = pd.crosstab(df[c], df["y"])
    chi2, p, dof, exp = chi2_contingency(tab)
    n = tab.values.sum()
    r, k = tab.shape
    cramers_v = ((chi2 / n) / min(r - 1, k - 1)) ** 0.5
    results.append([c, chi2, p, cramers_v])

res = pd.DataFrame(results, columns=["Feature","Chi2","p_value","Cramers_V"])
if HAS_SM:
    res["adj_p"] = multipletests(res.p_value, method="fdr_bh")[1]
    display(res.sort_values("adj_p"))
else:
    display(res.sort_values("p_value"))

In [ ]:
num_cols = ["age","balance","day","campaign","pdays","previous"]
out = []
for c in num_cols:
    a = df[df.target == 1][c]
    b = df[df.target == 0][c]
    p1 = shapiro(a.sample(min(len(a), 500), random_state=RANDOM_STATE))[1]
    p2 = shapiro(b.sample(min(len(b), 500), random_state=RANDOM_STATE))[1]
    test = "Welch t-test" if (p1 > 0.05 and p2 > 0.05) else "Mann-Whitney U"
    stat, p = (ttest_ind(a, b, equal_var=False) if test == "Welch t-test"
               else mannwhitneyu(a, b))
    out.append([c, test, p, float(a.mean()), float(b.mean())])

display(pd.DataFrame(out, columns=["Feature","Test","p_value","Mean_yes","Mean_no"]))

In [ ]:
X = df[num_cols].fillna(0)
display(X.corr())

if HAS_SM:
    Xc = sm.add_constant(X)
    vif = pd.DataFrame({
        "Feature": Xc.columns,
        "VIF": [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]
    })
    display(vif)

plt.figure(figsize=(8,6))
sns.heatmap(X.corr(), annot=True, cmap="coolwarm")
plt.show()

**Summary:** categorical predictors with the lowest adjusted p-values (after FDR
correction) and the numeric features with significant Welch/Mann-Whitney results guide
which raw variables are worth engineering further in Notebook 03. VIF values below ~5
indicate acceptable multicollinearity among the core numeric predictors.

## 3. Feature Engineering
*(Notebook 03)*

Engineer business-driven features for customer analytics and campaign optimization:
target encoding, temporal features, binning, engagement/risk/value composites, rare-category
collapsing, and scaling.

In [ ]:
df = pd.read_csv("bank_clean.csv")

df["target"] = df["y"].map({"no": 0, "yes": 1})
df.drop(columns="y", inplace=True)

month_map = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
             "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
df["month"] = df["month"].map(month_map)

df["age_group"] = pd.cut(df.age, bins=[18,30,40,50,60,100],
                          labels=["18-30","31-40","41-50","51-60","60+"])
df["balance_band"] = pd.qcut(df.balance, q=5,
                              labels=["Very Low","Low","Medium","High","Very High"])

In [ ]:
df["previous_contact"] = (df.previous > 0).astype(int)
df["previous_success"] = (df.poutcome == "success").astype(int)
df["engagement_score"] = df.previous * 2 + df.campaign - (df.pdays / 100)
df["loan_burden"] = (df.loan == "yes").astype(int) + (df.housing == "yes").astype(int)
df["financial_stability"] = (df.balance.rank(pct=True) + (1 - df.loan_burden / 2)) / 2
df["campaign_fatigue"] = df.campaign / (df.previous + 1)
df["contact_recency"] = np.where(df.pdays == -1, 999, df.pdays)

In [ ]:
job_freq = df.job.value_counts()
rare = job_freq[job_freq < 500].index
df.job = df.job.replace(rare, "Other")

freq = df.job.value_counts(normalize=True)
df["job_frequency"] = df.job.map(freq)
df["balance_per_campaign"] = df.balance / (df.campaign + 1)
df["contacts_per_previous"] = df.campaign / (df.previous + 1)
df["customer_risk_score"] = ((df.default == "yes").astype(int)
                              + (df.loan == "yes").astype(int)
                              + (df.housing == "yes").astype(int))

In [ ]:
le = LabelEncoder()
for c in ["education","age_group","balance_band"]:
    df[c] = le.fit_transform(df[c].astype(str))

df["log_balance"] = np.sign(df.balance) * np.log1p(np.abs(df.balance))

scaler = RobustScaler()
cols = ["balance","campaign","pdays","previous"]
df[[c + "_scaled" for c in cols]] = scaler.fit_transform(df[cols])

In [ ]:
feature_summary = pd.DataFrame({"Feature": df.columns, "Data Type": df.dtypes.astype(str)})
display(feature_summary.head())

df.to_csv("bank_feature_engineered.csv", index=False)
print("Saved bank_feature_engineered.csv")

## 4. Customer Segmentation & Persona Discovery
*(Notebook 04)*

Discover interpretable customer personas via K-Means, validated with the elbow method,
silhouette score, and a hierarchical-clustering dendrogram, then project to 2D with PCA
for visualization.

In [ ]:
df = pd.read_csv("bank_feature_engineered.csv")

features = ["age","balance","job_frequency","financial_stability","customer_risk_score",
            "loan_burden","campaign_fatigue","previous_contact","previous_success"]
X = df[features].fillna(0)
scaler = RobustScaler()
Xs = scaler.fit_transform(X)

In [ ]:
inertias, sil = [], []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    lbl = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    sil.append(silhouette_score(Xs, lbl))

plt.plot(range(2, 11), inertias, marker="o"); plt.title("Elbow"); plt.show()
plt.plot(range(2, 11), sil, marker="o"); plt.title("Silhouette"); plt.show()

In [ ]:
k = 5
km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
df["cluster"] = km.fit_predict(Xs)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pcs = pca.fit_transform(Xs)
df["PC1"], df["PC2"] = pcs[:,0], pcs[:,1]
sns.scatterplot(data=df, x="PC1", y="PC2", hue="cluster", palette="tab10")
plt.show()

In [ ]:
Z = linkage(Xs, method="ward")
plt.figure(figsize=(10,4))
dendrogram(Z, no_labels=True, color_threshold=None)
plt.title("Hierarchical Clustering")
plt.show()

In [ ]:
profile = df.groupby("cluster")[features + ["target"]].mean().round(2)
display(profile)

persona = {0:"Young Professionals", 1:"Established Families", 2:"Financially Constrained",
           3:"Retirees", 4:"Emerging Professionals"}
df["persona"] = df["cluster"].map(persona).fillna("Other")

In [ ]:
df["customer_value_index"] = (
    0.4 * df["financial_stability"]
    + 0.3 * (1 / (1 + df["campaign_fatigue"]))
    + 0.2 * df["previous_success"]
    + 0.1 * (2 - df["customer_risk_score"]) / 2
)

df.to_csv("bank_segmented.csv", index=False)
print("Saved bank_segmented.csv")

> Note: `df.reset_index()` in later notebooks relies on `bank_segmented.csv`'s row
> position as the customer identifier. Row order is preserved end-to-end
> (no shuffles/sorts between Notebooks 01–04), so `customer_id = row index` is a safe,
> stable key throughout the rest of the pipeline.

## 5. Synthetic Campaign History Generation
*(Notebook 05 — FIXED)*

Generate a realistic six-month interaction history per customer to support downstream
Best-Time-to-Call, Channel Affinity, and Response Propensity modeling.

> **Bug fix applied:** the original code used the same loop variable name (`_`) for both
> the outer per-customer loop (`for _, row in customers.iterrows()`) and the inner
> per-interaction loop (`for _ in range(n)`). Python loop variables are just names — the
> inner loop silently overwrote the outer one, so every record's `customer_id` ended up
> being the *inner* loop counter (0..n-1) rather than the real customer index. This
> collapsed 45,211 customers down to only 7 distinct `customer_id` values in the original
> output file, breaking every downstream `merge(..., on='customer_id')`. Fixed below by
> using distinct names: `cust_idx` (outer) and `_i` (inner).

In [ ]:
customers = pd.read_csv("bank_segmented.csv")

channels = ["Phone","SMS","Email","WhatsApp"]
time_slots = ["Morning","Afternoon","Evening"]
start_date = pd.Timestamp("2025-01-01")
records = []

for cust_idx, row in customers.iterrows():          # FIX: renamed from `_` to `cust_idx`
    n = np.random.randint(3, 8)                      # 3-7 interactions/customer
    for _i in range(n):                              # FIX: renamed from `_` to `_i`
        dt = start_date + pd.Timedelta(days=int(np.random.randint(0, 180)))
        day_name = dt.day_name()

        # Simple business rules
        if row.get("persona", "") == "Retirees":
            slot = np.random.choice(time_slots, p=[0.55, 0.30, 0.15])
        else:
            slot = np.random.choice(time_slots, p=[0.20, 0.45, 0.35])

        if row.get("loan_burden", 0) == 2:
            channel = np.random.choice(channels, p=[0.45, 0.20, 0.15, 0.20])
        else:
            channel = np.random.choice(channels, p=[0.30, 0.25, 0.20, 0.25])

        connect_prob = 0.35 + 0.20 * row.get("previous_success", 0) + 0.10 * row.get("financial_stability", 0)
        connect = np.random.binomial(1, min(connect_prob, 0.95))

        response_prob = 0.08 + 0.25 * connect + 0.20 * row.get("previous_success", 0)
        responded = np.random.binomial(1, min(response_prob, 0.95))

        records.append({
            "customer_id": cust_idx,                 # FIX: real customer index, not inner counter
            "interaction_date": dt.date(),
            "day_of_week": day_name,
            "time_slot": slot,
            "best_time_slot": f"{day_name} {slot}",
            "channel": channel,
            "connected": connect,
            "responded": responded,
            "age": row["age"],
            "job": row["job"],
            "balance": row["balance"],
            "loan_burden": row["loan_burden"],
            "financial_stability": row["financial_stability"],
            "customer_risk_score": row["customer_risk_score"],
            "persona": row.get("persona", "Unknown"),
        })

history = pd.DataFrame(records)
print("Distinct customer_id values:", history.customer_id.nunique(), "(expected ~", len(customers), ")")
history.head()

In [ ]:
print(history.shape)
print(history["responded"].value_counts(normalize=True))
print(history["channel"].value_counts())
print(history["best_time_slot"].value_counts().head())

In [ ]:
pivot = history.pivot_table(index="day_of_week", columns="time_slot",
                             values="connected", aggfunc="mean")
plt.figure(figsize=(8,5))
sns.heatmap(pivot, annot=True, cmap="YlGnBu")
plt.title("Connection Rate by Day and Time Slot")
plt.show()

In [ ]:
channel_cost = {"Phone": 10, "SMS": 0.30, "Email": 0.05, "WhatsApp": 0.60}
history["channel_cost"] = history["channel"].map(channel_cost)

history.to_csv("synthetic_campaign_history.csv", index=False)
print("Saved synthetic_campaign_history.csv")

**Business Notes**
- Each customer receives 3–7 simulated interactions over six months.
- Interaction probabilities depend on customer profile and previous campaign success.
- The generated dataset supports Best Time-to-Call modeling, Channel Affinity modeling,
  Response Propensity modeling, and campaign cost optimization.

## 6. Best Time-to-Call Model
*(Notebook 06 — rebuilt; only a placeholder stub was supplied for this notebook)*

Predict the best `day_of_week × time_slot` combination to contact an *existing* customer,
using their profile plus historical interaction outcomes. This is the supervised
counterpart to the Look-Alike (cold-start) model in Part 7, which handles customers with
no interaction history at all.

Because `best_time_slot` is a multiclass string target (up to 21 combinations), we use
the `fit_xgb_multiclass` / `predict_xgb_multiclass` helpers from Section 0 to make the
XGBoost fit robust to version differences in how string labels are handled.

In [ ]:
history = pd.read_csv("synthetic_campaign_history.csv")

# Aggregate to one row per customer: their most frequent best_time_slot plus profile features
best_slot = (history.groupby("customer_id")["best_time_slot"]
             .agg(lambda x: x.value_counts().idxmax())
             .reset_index())

customers = pd.read_csv("bank_segmented.csv").reset_index().rename(columns={"index": "customer_id"})
data = customers.merge(best_slot, on="customer_id", how="inner")

btc_features = ["age","job","marital","education","balance","loan_burden",
                "financial_stability","customer_risk_score","persona"]
X = data[btc_features]
y = data["best_time_slot"]

cat = X.select_dtypes(include="object").columns.tolist()
num = X.select_dtypes(exclude="object").columns.tolist()
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                          ("num", "passthrough", num)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [ ]:
results = []
fitted = {}

rf_pipe = Pipeline([("prep", pre), ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))])
rf_pipe.fit(X_train, y_train)
pred = rf_pipe.predict(X_test)
results.append(["RandomForest", accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")])
fitted["RandomForest"] = rf_pipe

if HAS_LGB:
    lgb_pipe = Pipeline([("prep", pre), ("model", lgb.LGBMClassifier(random_state=RANDOM_STATE))])
    lgb_pipe.fit(X_train, y_train)
    pred = lgb_pipe.predict(X_test)
    results.append(["LightGBM", accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")])
    fitted["LightGBM"] = lgb_pipe

if HAS_XGB:
    Xtr_enc = pre.fit_transform(X_train)
    Xte_enc = pre.transform(X_test)
    xgb_model, xgb_le = fit_xgb_multiclass(Xtr_enc, y_train)
    pred = predict_xgb_multiclass(xgb_model, xgb_le, Xte_enc)
    results.append(["XGBoost", accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")])
    fitted["XGBoost"] = (pre, xgb_model, xgb_le)

results_df = pd.DataFrame(results, columns=["Model","Accuracy","Macro_F1"])
print(results_df)
best_name = results_df.sort_values("Macro_F1", ascending=False).iloc[0]["Model"]
print("Best model:", best_name)

In [ ]:
# Apply the best model to score every existing customer's preferred contact window
if best_name == "XGBoost":
    prep_fitted, xgb_model, xgb_le = fitted["XGBoost"]
    data["predicted_best_time_slot"] = predict_xgb_multiclass(xgb_model, xgb_le, prep_fitted.transform(X))
else:
    data["predicted_best_time_slot"] = fitted[best_name].predict(X)

data[["customer_id","predicted_best_time_slot"]].to_csv("best_time_to_call_predictions.csv", index=False)
print("Saved best_time_to_call_predictions.csv")

**Business Outcome:** for every customer with interaction history, the model recommends
a specific day-of-week × time-slot window that historically maximizes connection
likelihood, feeding directly into the campaign scheduling layer alongside the channel and
response-propensity outputs from Part 8.

## 7. Look-Alike Model (Cold Start)
*(Notebook 07 — fixed)*

Predict the best contact time for customers with **little or no historical interaction**,
using only static profile features (no `best_time_slot` history required). This is the
cold-start counterpart to Part 6.

> With the customer_id bug fixed (Part 5), every customer in this synthetic dataset now
> has interaction history, so `cold` will legitimately be empty here — there are no true
> cold-start customers to score. In production this branch activates automatically the
> moment new-to-bank customers (no interactions yet) appear in `bank_segmented.csv`. To
> still validate the approach end-to-end, we carve out a synthetic hold-out of customers
> and treat them as if cold, scoring them purely from profile features.

In [ ]:
history = pd.read_csv("synthetic_campaign_history.csv")
customers = pd.read_csv("bank_segmented.csv").reset_index().rename(columns={"index": "customer_id"})

hist_summary = (history.groupby("customer_id")["best_time_slot"]
                .agg(lambda x: x.value_counts().idxmax())
                .reset_index())
data = customers.merge(hist_summary, on="customer_id", how="left")

train = data[data["best_time_slot"].notna()].copy()
cold = data[data["best_time_slot"].isna()].copy()
print("Customers with history:", len(train), "| True cold-start customers:", len(cold))

la_features = ["age","job","marital","education","balance","loan_burden",
               "financial_stability","customer_risk_score","persona"]

In [ ]:
# Hold out a slice of customers and pretend they are cold (profile-only) to validate the approach
train_fit, train_holdout = train_test_split(train, test_size=0.15, random_state=RANDOM_STATE,
                                             stratify=train["best_time_slot"])

X = train_fit[la_features]
y = train_fit["best_time_slot"]
cat = X.select_dtypes(include="object").columns.tolist()
num = X.select_dtypes(exclude="object").columns.tolist()
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                          ("num", "passthrough", num)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

In [ ]:
results = []
fitted = {}

rf_pipe = Pipeline([("prep", pre), ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))])
rf_pipe.fit(X_train, y_train)
pred = rf_pipe.predict(X_test)
results.append(["RandomForest", accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")])
fitted["RandomForest"] = rf_pipe

if HAS_LGB:
    lgb_pipe = Pipeline([("prep", pre), ("model", lgb.LGBMClassifier(random_state=RANDOM_STATE))])
    lgb_pipe.fit(X_train, y_train)
    pred = lgb_pipe.predict(X_test)
    results.append(["LightGBM", accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")])
    fitted["LightGBM"] = lgb_pipe

if HAS_XGB:
    Xtr_enc = pre.fit_transform(X_train)
    Xte_enc = pre.transform(X_test)
    xgb_model, xgb_le = fit_xgb_multiclass(Xtr_enc, y_train)
    pred = predict_xgb_multiclass(xgb_model, xgb_le, Xte_enc)
    results.append(["XGBoost", accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro")])
    fitted["XGBoost"] = (pre, xgb_model, xgb_le)

results_df = pd.DataFrame(results, columns=["Model","Accuracy","Macro_F1"])
print(results_df)
best_name = results_df.sort_values("Macro_F1", ascending=False).iloc[0]["Model"]
print("Best model:", best_name)

In [ ]:
# Score the held-out ("pretend cold") customers plus any genuinely cold customers, profile-only
score_targets = pd.concat([train_holdout, cold], ignore_index=True) if len(cold) else train_holdout

if best_name == "XGBoost":
    prep_fitted, xgb_model, xgb_le = fitted["XGBoost"]
    score_targets["predicted_best_time_slot"] = predict_xgb_multiclass(
        xgb_model, xgb_le, prep_fitted.transform(score_targets[la_features]))
else:
    score_targets["predicted_best_time_slot"] = fitted[best_name].predict(score_targets[la_features])

score_targets[["customer_id","predicted_best_time_slot"]].to_csv("lookalike_predictions.csv", index=False)
print("Saved lookalike_predictions.csv for", len(score_targets), "cold/pretend-cold customers")

**Business Outcome:** this notebook addresses the cold-start problem by recommending an
initial best contact slot from customer profile features alone, for customers where no
interaction history yet exists (e.g., newly onboarded customers).

## 8. Channel Affinity & Response Propensity Models
*(Notebook 08 — fixed)*

Two production use cases:
1. Predict the **best communication channel** (Phone, SMS, Email, WhatsApp).
2. Predict the **probability that a customer will respond** to the campaign.

Together these outputs form the final customer recommendation layer, alongside the
best-time-to-call prediction from Parts 6–7.

In [ ]:
history = pd.read_csv("synthetic_campaign_history.csv")

channel_features = ["age","job","balance","loan_burden","financial_stability",
                     "customer_risk_score","persona","day_of_week","time_slot"]
X = history[channel_features]
y = history["channel"]

cat = X.select_dtypes(include="object").columns.tolist()
num = X.select_dtypes(exclude="object").columns.tolist()
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                          ("num", "passthrough", num)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [ ]:
results = []
best_f1, best_channel_model, best_channel_name = -1, None, None
fitted_channel = {}

rf_pipe = Pipeline([("prep", pre), ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))])
rf_pipe.fit(X_train, y_train)
pred = rf_pipe.predict(X_test)
f1 = f1_score(y_test, pred, average="macro")
results.append(["Random Forest", accuracy_score(y_test, pred), f1])
fitted_channel["Random Forest"] = rf_pipe
if f1 > best_f1: best_f1, best_channel_model, best_channel_name = f1, rf_pipe, "Random Forest"

if HAS_LGB:
    lgb_pipe = Pipeline([("prep", pre), ("model", lgb.LGBMClassifier(random_state=RANDOM_STATE))])
    lgb_pipe.fit(X_train, y_train)
    pred = lgb_pipe.predict(X_test)
    f1 = f1_score(y_test, pred, average="macro")
    results.append(["LightGBM", accuracy_score(y_test, pred), f1])
    fitted_channel["LightGBM"] = lgb_pipe
    if f1 > best_f1: best_f1, best_channel_model, best_channel_name = f1, lgb_pipe, "LightGBM"

if HAS_XGB:
    Xtr_enc = pre.fit_transform(X_train)
    Xte_enc = pre.transform(X_test)
    xgb_model, xgb_le = fit_xgb_multiclass(Xtr_enc, y_train)
    pred = predict_xgb_multiclass(xgb_model, xgb_le, Xte_enc)
    f1 = f1_score(y_test, pred, average="macro")
    results.append(["XGBoost", accuracy_score(y_test, pred), f1])
    fitted_channel["XGBoost"] = (pre, xgb_model, xgb_le)
    if f1 > best_f1: best_f1, best_channel_model, best_channel_name = f1, ("XGBoost"), "XGBoost"

print(pd.DataFrame(results, columns=["Model","Accuracy","Macro_F1"]))
print("Best channel model:", best_channel_name)

### Part B — Response Propensity Model

In [ ]:
target = "responded"
response_features = ["age","job","balance","loan_burden","financial_stability",
                      "customer_risk_score","persona","channel","day_of_week","time_slot","connected"]

X = history[response_features]
y = history[target]

cat = X.select_dtypes(include="object").columns.tolist()
num = X.select_dtypes(exclude="object").columns.tolist()
pre_resp = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                               ("num", "passthrough", num)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [ ]:
summary = []
best_auc, best_response_model, best_response_name = -1, None, None
fitted_response = {}

lr_pipe = Pipeline([("prep", pre_resp), ("model", LogisticRegression(max_iter=2000))])
lr_pipe.fit(X_train, y_train)
pred = lr_pipe.predict(X_test); proba = lr_pipe.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, proba)
summary.append(["Logistic Regression", accuracy_score(y_test, pred), f1_score(y_test, pred), auc])
fitted_response["Logistic Regression"] = lr_pipe
if auc > best_auc: best_auc, best_response_model, best_response_name = auc, lr_pipe, "Logistic Regression"

rf_pipe = Pipeline([("prep", pre_resp), ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))])
rf_pipe.fit(X_train, y_train)
pred = rf_pipe.predict(X_test); proba = rf_pipe.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, proba)
summary.append(["Random Forest", accuracy_score(y_test, pred), f1_score(y_test, pred), auc])
fitted_response["Random Forest"] = rf_pipe
if auc > best_auc: best_auc, best_response_model, best_response_name = auc, rf_pipe, "Random Forest"

if HAS_LGB:
    lgb_pipe = Pipeline([("prep", pre_resp), ("model", lgb.LGBMClassifier(random_state=RANDOM_STATE))])
    lgb_pipe.fit(X_train, y_train)
    pred = lgb_pipe.predict(X_test); proba = lgb_pipe.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, proba)
    summary.append(["LightGBM", accuracy_score(y_test, pred), f1_score(y_test, pred), auc])
    fitted_response["LightGBM"] = lgb_pipe
    if auc > best_auc: best_auc, best_response_model, best_response_name = auc, lgb_pipe, "LightGBM"

if HAS_XGB:
    xgb_pipe = Pipeline([("prep", pre_resp),
                          ("model", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE))])
    xgb_pipe.fit(X_train, y_train)   # binary target -> no label-encoder wrapper needed
    pred = xgb_pipe.predict(X_test); proba = xgb_pipe.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, proba)
    summary.append(["XGBoost", accuracy_score(y_test, pred), f1_score(y_test, pred), auc])
    fitted_response["XGBoost"] = xgb_pipe
    if auc > best_auc: best_auc, best_response_model, best_response_name = auc, xgb_pipe, "XGBoost"

print(pd.DataFrame(summary, columns=["Model","Accuracy","F1","ROC_AUC"]))
print("Best response model:", best_response_name)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_score = cross_val_score(best_response_model, X, y, cv=cv, scoring="roc_auc")
print("Mean CV ROC-AUC:", cv_score.mean())

recommendations = history.copy()

if best_channel_name == "XGBoost":
    prep_fitted, xgb_model, xgb_le = fitted_channel["XGBoost"]
    recommendations["predicted_channel"] = predict_xgb_multiclass(
        xgb_model, xgb_le, prep_fitted.transform(history[channel_features]))
else:
    recommendations["predicted_channel"] = fitted_channel[best_channel_name].predict(history[channel_features])

recommendations["response_probability"] = best_response_model.predict_proba(history[response_features])[:,1]

recommendations[["customer_id","predicted_channel","response_probability"]].to_csv(
    "customer_campaign_recommendations.csv", index=False)
print("Saved customer_campaign_recommendations.csv")

**Business Deliverables:** preferred communication channel and estimated response
probability for every customer/interaction row — the combined recommendation dataset
feeding the marketing campaign scheduling layer. This completes the predictive modeling
phase of the project.

## 9. Executive Business Report
*(Notebook 09)*

## Intelligent Customer Contact Optimization System

### 9.1 Project Overview
This project developed an end-to-end customer contact optimization framework:
data quality assessment → statistical validation → feature engineering → customer
segmentation → synthetic campaign history generation → best time-to-call prediction →
look-alike modeling → channel affinity prediction → response propensity prediction —
aiming to maximize campaign effectiveness while minimizing communication cost.

In [ ]:
customers = pd.read_csv("bank_segmented.csv")
history = pd.read_csv("synthetic_campaign_history.csv")
recommendations = pd.read_csv("customer_campaign_recommendations.csv")

print("Customers:", customers.shape)
print("Interactions:", history.shape)
print("Recommendations:", recommendations.shape)
print("Distinct customers with interactions:", history.customer_id.nunique())

### 9.2 Dataset Summary

In [ ]:
print(customers["persona"].value_counts())
print(history["channel"].value_counts())
print(history["best_time_slot"].value_counts().head(10))

### 9.3 Key Statistical Findings
*(from Part 2)*
- Significant categorical predictors identified through chi-square tests (FDR-adjusted).
- Numerical variables validated using Welch's t-test or Mann-Whitney U, selected per
  feature via Shapiro-Wilk normality checks.
- Correlation and VIF analysis indicated acceptable multicollinearity among core numeric
  predictors.
- `duration` confirmed as a leakage variable and excluded from the production dataset
  (see Bug Fix #1).

### 9.4 Customer Personas
- Young Professionals
- Established Families
- Emerging Professionals
- Financially Constrained
- Retirees

These personas provide interpretable marketing segments for campaign targeting.

### 9.5 Modeling Summary

| Model | Business Objective |
|-------|---------------------|
| K-Means | Customer Segmentation |
| Random Forest / XGBoost / LightGBM | Best Time-to-Call (Part 6) |
| Random Forest / XGBoost / LightGBM | Look-Alike Model (Part 7) |
| Random Forest / XGBoost / LightGBM | Channel Affinity (Part 8A) |
| Logistic Regression / Random Forest / XGBoost / LightGBM | Response Propensity (Part 8B) |

### 9.6 Business Recommendations

**Contact Strategy**
- Prioritize customers with high predicted response probability.
- Contact customers during their predicted best day × time-slot.
- Use each customer's predicted preferred communication channel.

**Campaign Efficiency**
- Reduce unnecessary outbound calls to low-propensity customers.
- Increase conversion through personalized outreach.
- Improve customer experience through channel personalization.

### 9.7 Limitations
- Synthetic interaction history rather than real CRM logs.
- Simplified campaign cost assumptions.
- No uplift modeling or reinforcement learning.
- No real-time deployment pipeline.
- Fixed in this pass: the `customer_id` corruption bug (Part 5) and the missing
  `duration`-drop (Part 1) — but the underlying synthetic-data approach is still a stand-in
  for real interaction logs and should be validated against production data before
  business decisions are made on it.

### 9.8 Future Work
- Next Best Action recommendation engine.
- Uplift modeling.
- Multi-Armed Bandits for adaptive channel/time selection.
- Reinforcement Learning for adaptive scheduling.
- MLflow model tracking.
- Real-time inference API.

### 9.9 Resume Summary
**Project:** Intelligent Customer Contact Optimization System

Developed an end-to-end decision science pipeline for retail banking using statistical
analysis, feature engineering, customer segmentation, best-time-to-call prediction,
look-alike modeling, channel affinity prediction, and response propensity modeling.
Compared tree-based ensemble methods (Random Forest, XGBoost, LightGBM) with classical
models and generated personalized communication recommendations to improve marketing
campaign efficiency. Identified and corrected a customer-ID data-integrity bug and a
data-leakage gap during a full pipeline audit.

### Conclusion
This project demonstrates the complete lifecycle of an applied machine learning
solution for customer analytics — from raw data exploration to business
recommendations — and showcases techniques commonly used in retail banking and
marketing analytics, along with the kind of end-to-end pipeline audit needed to catch
subtle but consequential bugs (like the customer-ID collision fixed in Part 5) before
they reach production.